In [1]:
import numpy as np
from astropy.io import fits
from astropy.wcs import WCS
from astropy.coordinates import SkyCoord
import astropy.units as u
import mysql.connector
import pandas as pd
import os

In [2]:
mydb = mysql.connector.connect(
  host="127.0.0.1",
  user="root",
  #passwd=" OpenSQL1000",        #<---- 
  database = "HiGAL_250_280"     #<---- 
)

In [3]:
def extract(cube_file, output_dir):
    # Open the spectral cube FITS file
    with fits.open(cube_file,ignore_missing_simple=True) as hdul:
        cube_data = hdul[0].data
        header = hdul[0].header
        wcs = WCS(header)

    # Create SQL engine and read coordinates -- crates data frame with all clumps within the cube
	# This is set for the 1 degree cubes. The range will need to be changed if bigger
    df = pd.read_sql_query(f'SELECT gal_long, gal_lat, higal_name FROM higal_mopra_co WHERE gal_long>{int(cube_file[1:4])} and gal_long<{int(cube_file[1:4])+10}', con=mydb)

    for index, row in df.iterrows():
        # Convert galactic coordinates to pixel coordinates
        sky_coord = SkyCoord(l=row['gal_long']*u.deg, b=row['gal_lat']*u.deg, frame='galactic')
        x, y = sky_coord.to_pixel(wcs, origin=0)

        # Define the 3x3 pixel area around the source
        x_min_spec, x_max_spec = int(x) - 1, int(x) + 1
        y_min_spec, y_max_spec = int(y) - 1, int(y) + 1

        # Ensure we don't go out of bounds for the spectra
        x_min_spec = max(x_min_spec, 0)
        y_min_spec = max(y_min_spec, 0)
        x_max_spec = min(x_max_spec, cube_data.shape[2] - 1)
        y_max_spec = min(y_max_spec, cube_data.shape[1] - 1)

        # Extract the smaller cube for the spectra
        subcube_spec = cube_data[:, y_min_spec:y_max_spec+1, x_min_spec:x_max_spec+1]

        # Compute the spectra (average over the 3x3 area)
        spectra = np.mean(subcube_spec, axis=(1, 2))

        # Create a new FITS file for the spectra
        hdu_spec = fits.PrimaryHDU(spectra, header)
        output_path_spec = f"{output_dir}/12CO_{row['higal_name']}_spec.fits"
       # hdu_spec.writeto(output_path_spec, overwrite=True)
        #print(f"Saved spectra to {output_path_spec}")


In [ ]:
output_dir = '/Users/kyriakos.trakakis/Desktop/Temp'		# output directory
cube_folder = os.listdir('/Users/kyriakos.trakakis/Downloads/MOPRA_HiGAL' )	  	# directory of cubes
# spectra_folder.remove('.DS_Store')
cube_folder.sort()

In [26]:
## Ignore this entirely, 

path = "/Users/kyriakos.trakakis/Downloads/Other_stuff/"
filename = "lowrel_HiGAL_360_clump_catalogue.tab"

# Extract column names from the first line
with open(path + filename, "r") as f:
    header = f.readline()

columns = [
    col.strip()
    for col in header.strip().strip("|").split("|")
]

# Read the fixed-width data, skipping the 3 header lines
df = pd.read_fwf(
    path + filename,
    skiprows=3,
    names=columns
)